# 02 Retrieval Experiments

Notebook này dùng để so sánh các phương pháp retrieval cho Vietnamese Labor Legal RAG.

Mục tiêu:
- chạy cùng một bộ câu hỏi test cho nhiều method,
- đo `Recall@5`, `MRR`, `nDCG@5`, `citation_coverage`,
- chọn phương án cân bằng để dùng cho app/deploy sau này.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.evaluation.evaluate_retrieval import evaluate_methods

CORPUS_PATH = PROJECT_ROOT / "data/processed/labor_corpus_sample.jsonl"

## Fast comparison

`use_tfidf_fallback=True` giúp notebook chạy nhanh trên máy yếu. Khi muốn benchmark thật bằng embedding + FAISS, đổi thành `False`.

In [ ]:
report = evaluate_methods(str(CORPUS_PATH), use_tfidf_fallback=True)
rows = []
for method, metrics in report["methods"].items():
    rows.append({
        "method": method,
        "recall@5": metrics["recall@5"],
        "mrr": metrics["mrr"],
        "ndcg@5": metrics["ndcg@5"],
        "citation_coverage": metrics["citation_coverage"],
    })

df = pd.DataFrame(rows)
df

In [ ]:
score_weights = {"recall@5": 0.45, "mrr": 0.35, "ndcg@5": 0.20}
df["balanced_score"] = (
    score_weights["recall@5"] * df["recall@5"]
    + score_weights["mrr"] * df["mrr"]
    + score_weights["ndcg@5"] * df["ndcg@5"]
)
df.sort_values("balanced_score", ascending=False)

In [ ]:
ax = df.set_index("method")[["recall@5", "mrr", "ndcg@5"]].plot(kind="bar", ylim=(0, 1.05), figsize=(10, 4))
ax.set_title("Retrieval method comparison")
ax.set_ylabel("score")

## Selected method for app/deploy

Default method: **`hybrid_rrf`**.

Lý do chọn:
- BM25 mạnh với thuật ngữ pháp lý chính xác.
- Vector/FAISS mạnh với câu hỏi tự nhiên.
- RRF fusion ổn định hơn weighted-score fusion vì không phụ thuộc scale điểm BM25/vector.
- MMR chỉ bật khi context bị lặp nhiều; mặc định không bật để giữ ranking đơn giản và dễ giải thích.